### This version works with files from Jan 17th

This script is a work in progress, and atm does not work properly with the given files

In [12]:
import numpy as np
import uproot
import matplotlib as mpl
import os
import datetime as dt
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from itertools import islice



import json
from urllib.parse import urlencode
from urllib.request import urlopen, Request


In [15]:
# splits log file into chunks based on position of json config files
# first chunk is junk
# first line of chunk is path to json config file
# second line of chunk is the contents of the json file
# file path - path with lines of data
# chunks - array of arrays
def split_log_by_json(filepath):
    chunks = []
    current_chunk = []

    with open(filepath, "r") as f:
        for line in f:
            if ".json" in line:
                if current_chunk:
                    chunks.append(current_chunk)
                current_chunk = [line]
            else:
                current_chunk.append(line)

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

In [16]:
format_pattern = '%Y-%m-%d %H:%M:%S.%f'

In [19]:
log_file = 'S89_Jan17.log' # Manually input

chunk_lst = split_log_by_json(log_file)

# need list of keywords that are in .raw file name
raw_keywords = ['string', 'device', 'port', 'cam', 'illum', 'gain', 'exposure']

# need list of keywords that are in other JSON DB files but not present in log files
# from Jan17th, excluding RETRY
json_keywords = ['LED_ON', 'LED_OFF', 'CAPTURE_START', 'CAPTURE_FINISH']

result  = [] # Dictionary that will be transformed into .json

for i, chunk in enumerate(chunk_lst):
    if ".json" in chunk[0]:
        json_fname = chunk[0].strip()
        json_lst = chunk[2].strip()
        contents = chunk[1:]
        for j, line in enumerate(contents):
            if ".raw" in line:
                # Data unavailable in log file
                d = {}
                d["RETRY"] = "Unknown"
                raw_fname = line.split(" ")[-1].strip() # raw file name
                d["FILENAME"] = raw_fname
                for key in json_keywords:
                    d[key] = "Unknown"
                
                # Scrape data from raw file name
                if raw_fname != "None":
                    raw_split = raw_fname.strip().split('_')
                    run_type = '_'.join(raw_split[1:-8])
                    d['run_type'] = run_type
                    for a, data in enumerate(raw_split[-8:-1]):
                            d[raw_keywords[a]] = data.strip().replace(raw_keywords[a], '')      
                else:
                    d['run_type'] = "None"
                    for a in range(len(raw_keywords)):
                        d[raw_keywords[a]] = "None"

                d["config_fname"] = json_fname
                d["config_lst"] = json_lst
                result.append(d)

# Write result to .json File
output_json_name = log_file.replace(".log", "_parsed.json")
with open(output_json_name, "w") as f:
    json.dump(result, f, indent=4)